# Starter notebook: where should the food bank listen first?

This notebook loads the HAI civic participation files and builds one example score. Change the weights, then explain your choice in plain language.

Rules from the challenge page:

* Score ranked rows only. Not-ranked rows stay grey.
* Use up to 3 columns. P1 counts for at least half, civic organizations for a quarter or less.
* Leave the five neighbourhood conditions out of your score. They are background only.
* The shortlist is for listening (focus groups and outreach conversations), not for voter registration or get-out-the-vote.
* Your ZIP-level shortlist stays in your team folder. Slides and the live demo use county views only.

## 1. Get the files

In Colab, run the next cell and upload `zcta_civic.csv` and `county_civic.csv`. On your own computer, put both files in the same folder as this notebook.

In [ ]:
try:
    from google.colab import files  # only exists in Colab
    files.upload()
except ImportError:
    pass

## 2. Load them

Row 1 of each CSV is a note on how the data may be used, so it is skipped. ZIP codes and county codes are read as text so leading zeros are kept.

In [ ]:
from pathlib import Path

import pandas as pd

DATA_DIR = Path(".")
TEXT = {"ZCTA": str, "county_main": str, "county_fips": str}

zcta = pd.read_csv(DATA_DIR / "zcta_civic.csv", skiprows=1, dtype=TEXT)
county = pd.read_csv(DATA_DIR / "county_civic.csv", skiprows=1, dtype=TEXT)

zcta["not_ranked"] = zcta["not_ranked"].astype(str).str.lower().isin(["true", "1"])
can_rank = ~zcta["not_ranked"]
print(f"{len(zcta)} ZIP areas: {int(can_rank.sum())} ranked, {int((~can_rank).sum())} not ranked")

## 3. County view first

County numbers are exact and safe for slides. `P1_low` and `P1_high` give the range. Counties are listed alphabetically, not by turnout.

In [ ]:
county_view = county[["county_label", "P1", "P1_low", "P1_high", "zctas_total", "zctas_ranked"]]
print(county_view.sort_values("county_label").to_string(index=False))

## 4. Your score

`WEIGHTS` is the part to change. Each column is turned into a percentile among ranked rows (0 = lowest, 1 = highest), then the weights are applied. A low score means lower measured participation: a place to listen first. A high score means a place to work with.

`reg_rate` is registered voters in 2024 divided by adult citizens. It is an example third column.

In [ ]:
WEIGHTS = {"P1": 0.6, "civic_orgs_n": 0.2, "reg_rate": 0.2}

ALLOWED = {"P1", "civic_orgs_n", "reg_rate", "to_reg24"}
assert len(WEIGHTS) <= 3, "Use up to 3 columns."
assert set(WEIGHTS) <= ALLOWED, "Leave the five neighbourhood conditions out of the score."
assert WEIGHTS.get("P1", 0) >= 0.5 * sum(WEIGHTS.values()), "P1 should count for at least half."
assert WEIGHTS.get("civic_orgs_n", 0) <= 0.25 * sum(WEIGHTS.values()), "Civic orgs: a quarter or less."

ranked = zcta[can_rank].copy()
ranked["reg_rate"] = ranked["reg24"] / ranked["cvap_adj"]


def score(rows, weights):
    total = sum(weights.values())
    parts = [rows[col].rank(pct=True) * w for col, w in weights.items()]
    return sum(parts) / total


ranked["score"] = score(ranked, WEIGHTS)
print(ranked["score"].describe().round(3).to_string())

## 5. A shortlist of 8 across at least 4 counties

Four places from each end. The pick spreads across counties before repeating one. Rows with a caution flag stay in but are shown so you can say what to watch.

In [ ]:
SHOW = ["ZCTA", "county_label", "score", "P1", "civic_orgs_n", "caution"]


def pick(rows, n, lowest):
    """n rows from one end of the score, one per county before any county repeats."""
    order = rows.sort_values("score", ascending=lowest)
    first = order.drop_duplicates("county_label")
    rest = order.drop(index=first.index)
    return pd.concat([first, rest]).head(n)


listen_first = pick(ranked, 4, lowest=True).assign(group="listen first")
work_with = pick(ranked.drop(index=listen_first.index), 4, lowest=False).assign(group="work with")
shortlist = pd.concat([listen_first, work_with])

n_counties = shortlist["county_label"].nunique()
print(f"Shortlist covers {n_counties} counties.")
if n_counties < 4:
    print("Fewer than 4 counties: adjust by hand, or use county level where ZIP areas are too small.")
print("Check that it includes both city and rural places.")
print(shortlist[["group", *SHOW]].to_string(index=False))

## 6. Does it hold up with a second weighting?

In [ ]:
ALT_WEIGHTS = {"P1": 0.8, "civic_orgs_n": 0.1, "reg_rate": 0.1}
alt = ranked.assign(score=score(ranked, ALT_WEIGHTS))
alt_low = pick(alt, 4, lowest=True)
alt_list = pd.concat([alt_low, pick(alt.drop(index=alt_low.index), 4, lowest=False)])
kept = len(set(shortlist.index) & set(alt_list.index))
print(f"{kept} of {len(shortlist)} places stay on the shortlist with the second weighting.")

## 7. Save for your team folder

This file names ZIP areas. It goes in your team folder for the food bank, never on slides, posts or a public repo.

In [ ]:
shortlist[["group", *SHOW]].to_csv("team_folder_shortlist.csv", index=False)
print("Saved team_folder_shortlist.csv")